<a href="https://colab.research.google.com/github/Amper2B/GVC_MiniCaseStudy/blob/main/FusionModelv03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm

# =====================================================================
# SYSTEM CONFIGURATION & GPU DESTINATION DISCOVERY
# =====================================================================
# Automatically configures execution on T4 GPU if activated in Colab settings
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== SYSTEM HARDWARE INITIALIZATION ===")
print(f"Active processing infrastructure destination: {device}\n")

# =====================================================================
# STEP 1: INTERACTIVE DIRECTORY BUILDER AND CLEAN UPLOADER
# =====================================================================
# Finalized 10 target classes matching your custom mobile dataset
class_names = ["boy", "corn", "coconut", "car", "monument",
               "dog", "face", "eye", "softdrink", "penguin"]

# Build isolated structural directories on the cloud file system
for split in ['train', 'valid']:
    for name in class_names:
        os.makedirs(os.path.join('dataset', split, name), exist_ok=True)

print("=== STEP 1: INTERACTIVE DIRECTORY BUILDER ===")
print("Created directory layers inside 'dataset/train/' and 'dataset/valid/'.")

# Interactive console loop designed for mobile photo uploads
while True:
    mode = input("\nDo you want to upload images? (type 'yes' to upload or 'no' to start training): ").strip().lower()
    if mode == 'no':
        break
    if mode != 'yes':
        print("Invalid input. Please enter 'yes' or 'no'.")
        continue

    chosen_split = input("Enter split destination ('train' or 'valid'): ").strip().lower()
    if chosen_split not in ['train', 'valid']:
        print("Invalid partition selection. Choose 'train' or 'valid'.")
        continue

    print("\nAvailable Target Categories:")
    for i, name in enumerate(class_names):
        print(f"[{i}] {name}")

    try:
        choice = int(input("\nEnter class index number to populate: "))
        target_class = class_names[choice]
    except (ValueError, IndexError):
        print("Invalid index choice. Select a valid number from the list.")
        continue

    target_path = os.path.join('dataset', chosen_split, target_class)
    print(f"\nSelect files from your phone gallery for: {target_path}")

    # Trigger the browser file selector
    uploaded = files.upload()

    # Process the files with a single clean loading progress bar
    if uploaded:
        file_list = list(uploaded.keys())
        print(f"\nProcessing {len(file_list)} uploaded files...")

        # tqdm creates a single updating bar and removes line-by-line spam
        for file_name in tqdm(file_list, desc=f"Moving to {target_class}", leave=False):
            destination = os.path.join(target_path, file_name)
            os.rename(file_name, destination)

        print(f"✅ Successfully committed all assets to: {target_path}")

# =====================================================================
# STEP 2: DATA PREPROCESSING & AUGMENTATION PIPELINE
# =====================================================================
print("\n=== STEP 2: DATA PREPROCESSING PIPELINE ===")

# Augmentations regularize learning bounds for custom datasets
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

try:
    train_dataset = ImageFolder('dataset/train', transform=train_transform)
    val_dataset = ImageFolder('dataset/valid', transform=val_transform)
except FileNotFoundError:
    print("Error: Target directories are unpopulated. Run cell again to upload images.")
    raise

num_classes = len(train_dataset.classes)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Dynamic Categories Found : {train_dataset.classes}")
print(f"Training Samples Batched : {len(train_dataset)}")
print(f"Validation Samples Loaded: {len(val_dataset)}")

# =====================================================================
# STEP 3: DEFINE THE DUAL-BACKBONE FUSION ARCHITECTURE
# =====================================================================
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        # Extract configurations pre-trained on ImageNet
        self.resnet = models.resnet18(weights='DEFAULT')
        self.mobile = models.mobilenet_v2(weights='DEFAULT')

        # Drop terminal classification heads using Identity links
        self.resnet.fc = nn.Identity()
        self.mobile.classifier = nn.Identity()

        # Input dimension sums feature widths: 512 (ResNet) + 1280 (MobileNet) = 1792
        self.classifier = nn.Linear(1792, num_classes)

    def forward(self, x):
        f1 = self.resnet(x)
        f2 = self.mobile(x)
        combined = torch.cat((f1, f2), dim=1) # Parallel feature concatenation
        return self.classifier(combined)

# Build and map model memory spaces directly onto the active hardware accelerator
model = FusionModel(num_classes=num_classes).to(device)

# =====================================================================
# STEP 4: MODEL OPTIMIZATION PIPELINE
# =====================================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_losses = []
val_losses = []

print("\n=== STEP 4: TRAINING CONCATENATED FUSION PIPELINE ===")
for epoch in range(15):
    # Active training execution
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device) # GPU migration

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    # Active validation check to capture metrics without leakage
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device) # GPU migration
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

    avg_train = running_train_loss / len(train_loader)
    avg_val = running_val_loss / len(val_loader)

    train_losses.append(avg_train)
    val_losses.append(avg_val)

    print(f"Epoch {epoch+1:02d}/15 | Train Loss: {avg_train:.4f} | Validation Loss: {avg_val:.4f}")

# =====================================================================
# STEP 5: GRAPH VISUALIZATION: CONVERGENCE PROFILE LOSS CURVE
# =====================================================================
plt.figure(figsize=(10, 5))
plt.plot(range(1, 16), train_losses, marker='o', color='blue', linewidth=2, label='Training Loss')
plt.plot(range(1, 16), val_losses, marker='s', color='red', linewidth=2, label='Validation Loss')
plt.title("Convergence Profile: Training vs. Validation Loss", fontsize=12, fontweight='bold')
plt.xlabel("Training Epoch", fontsize=10)
plt.ylabel("Cross-Entropy Loss", fontsize=10)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--')
plt.show()

# =====================================================================
# STEP 6: METRIC EVALUATION: NORMALIZED CONFUSION MATRIX
# =====================================================================
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Generate row-normalized accuracy rates for standard LNCS profiles
cm = confusion_matrix(all_labels, all_preds, normalize='true')

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)
plt.title("Normalized Confusion Matrix (Validation Partition Data)", fontsize=12, fontweight='bold')
plt.ylabel('Actual Object Class', fontsize=10)
plt.xlabel('Predicted Object Class', fontsize=10)
plt.show()

print("\n=== CLASSIFICATION REPORT (VALIDATION PARTITION) ===")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

# =====================================================================
# STEP 7: TRAINING STANDALONE BASELINE MODEL (ResNet18 Only)
# =====================================================================
baseline_model = models.resnet18(weights='DEFAULT')
baseline_model.fc = nn.Linear(512, num_classes)
baseline_model = baseline_model.to(device) # GPU migration
b_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)

print("\n=== STEP 7: TRAINING BASELINE DATA PIPELINE ===")
for epoch in range(15):
    baseline_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device) # GPU migration
        b_optimizer.zero_grad()
        outputs = baseline_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        b_optimizer.step()
print("Baseline Training Complete.")

# =====================================================================
# STEP 8: PERFORMANCE ACCURACY BENCHMARKING FUNCTION
# =====================================================================
def calculate_accuracy(test_model, data_loader):
    test_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, lbls in data_loader:
            imgs, lbls = imgs.to(device), lbls.to(device) # GPU migration
            out = test_model(imgs)
            _, predicted = torch.max(out.data, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return (correct / total) * 100

base_acc = calculate_accuracy(baseline_model, val_loader)
fusion_acc = calculate_accuracy(model, val_loader)
improvement = fusion_acc - base_acc

# =====================================================================
# STEP 9: GRAPH VISUALIZATION: MODEL COMPARISON BAR CHART
# =====================================================================
plt.figure(figsize=(7, 5))
bars = plt.bar(['Baseline\n(ResNet18)', 'Proposed Fusion\n(ResNet18+MobileNetV2)'],
               [base_acc, fusion_acc], color=['darkgray', 'royalblue'], width=0.5)
plt.ylabel('Validation Accuracy (%)', fontsize=10)
plt.title('Empirical Performance Comparison', fontsize=12, fontweight='bold')
plt.ylim(0, 110)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Overlay exact boundary numbers directly on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.show()

print("\n=== EMPIRICAL VARIANCE REPORT ===")
print(f"Final Baseline Validation Accuracy: {base_acc:.2f}%")
print(f"Final Proposed Fusion Accuracy    : {fusion_acc:.2f}%")
print(f"Calculated Empirical Improvement   : {improvement:.2f}%\n")

# =====================================================================
# STEP 10: SERIALIZATION AND WEIGHTS EXPORT PREPARATION
# =====================================================================
print("=== STEP 10: LOCAL SERIALIZATION EXPORT ===")
# Transfer parameter dict onto host CPU memory before storage serialization
torch.save(model.to('cpu').state_dict(), 'fusion_model_weights.pth')
files.download('fusion_model_weights.pth')
print("Model weights exported and downloaded successfully.")